# Notebook 01 — Descarga y EDA de Banco Sabadell (SAB.MC)

**Objetivo:** Descargar 5 años de cotización de Banco Sabadell, hacer un análisis exploratorio inicial y guardar el dataset limpio para los notebooks posteriores.

**Contexto del activo:** Banco Sabadell (BME: SAB) — quinto banco español por activos. Periodo cubierto: 2021–2026. Marca el contexto de la OPA hostil de BBVA anunciada el 30/04/2024.


## 1. Imports

In [ ]:
import sys
from pathlib import Path

# Para poder importar desde src/ aunque ejecutemos el notebook desde notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from src.data_loader import get_sab, OPA_BBVA_EVENTS

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)


## 2. Descarga del histórico (5 años)

In [ ]:
# Descarga con caché en data/. force_refresh=True solo la primera vez o si quieres datos nuevos.
sab = get_sab(period="5y", force_refresh=False)
sab.head()


In [ ]:
# Verificación básica
print("Filas:", len(sab))
print("Rango:", sab["Date"].min().date(), "->", sab["Date"].max().date())
print("Columnas:", list(sab.columns))
sab.info()


In [ ]:
# Estadística descriptiva del cierre
sab[["Open", "High", "Low", "Close", "Volume"]].describe().T


## 3. Limpieza y feature engineering básico

In [ ]:
# Eliminar filas con valores nulos por si los hubiera
print("Nulos por columna:")
print(sab.isnull().sum())

sab = sab.dropna().reset_index(drop=True)


In [ ]:
# Features útiles para análisis posterior:
sab["Return"]    = sab["Close"].pct_change()
sab["LogReturn"] = np.log(sab["Close"] / sab["Close"].shift(1))
sab["SMA_20"]    = sab["Close"].rolling(20).mean()
sab["SMA_50"]    = sab["Close"].rolling(50).mean()
sab["Vol_20"]    = sab["Return"].rolling(20).std() * np.sqrt(252)  # volatilidad anualizada
sab.tail()


## 4. EDA visual

In [ ]:
# Serie de cierre con medias móviles
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sab["Date"], sab["Close"],  label="Close",  linewidth=1.5)
ax.plot(sab["Date"], sab["SMA_20"], label="SMA 20", linewidth=1, alpha=0.7)
ax.plot(sab["Date"], sab["SMA_50"], label="SMA 50", linewidth=1, alpha=0.7)

# Línea vertical marcando el anuncio de la OPA
for ev_date in OPA_BBVA_EVENTS["ds"]:
    ax.axvline(ev_date, color="red", linestyle="--", alpha=0.3)

ax.set_title("Banco Sabadell — Cierre + medias móviles (eventos OPA BBVA marcados)")
ax.set_xlabel("Fecha"); ax.set_ylabel("EUR")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Volumen
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(sab["Date"], sab["Volume"], width=1, color="steelblue", alpha=0.6)
ax.set_title("Volumen diario negociado")
ax.set_xlabel("Fecha"); ax.set_ylabel("Acciones")
plt.tight_layout()
plt.show()


In [ ]:
# Distribución de retornos diarios
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(sab["Return"].dropna(), bins=80, kde=True, ax=axes[0])
axes[0].set_title("Distribución de retornos diarios")
axes[0].axvline(0, color="black", linestyle="--", alpha=0.5)

axes[1].plot(sab["Date"], sab["Vol_20"], color="darkorange")
axes[1].set_title("Volatilidad rolling 20d (anualizada)")
plt.tight_layout()
plt.show()


In [ ]:
# Métricas resumen
print(f"Retorno medio diario : {sab['Return'].mean()*100:.4f} %")
print(f"Volatilidad anualizada: {sab['Return'].std()*np.sqrt(252)*100:.2f} %")
print(f"Sharpe (rf=0)        : {(sab['Return'].mean()/sab['Return'].std())*np.sqrt(252):.3f}")
print(f"Máximo retorno diario: {sab['Return'].max()*100:.2f} %")
print(f"Peor retorno diario  : {sab['Return'].min()*100:.2f} %")


## 5. Exploración interactiva (opcional, como en clase)

In [ ]:
# Mismo patrón que tu analisis_de_datos.ipynb — pygwalker
# Si no lo tienes instalado: !pip install pygwalker
try:
    import pygwalker as pyg
    walker = pyg.walk(sab)
except ImportError:
    print("pygwalker no instalado. Saltamos.")


In [ ]:
# Informe automático con dataprep (mismo patrón de limpieza_datos.ipynb)
try:
    from dataprep.eda import create_report
    report = create_report(sab.drop(columns=["Date"]))
    report.save("../data/eda_sab.html")
    print("Informe EDA guardado en data/eda_sab.html")
except ImportError:
    print("dataprep no instalado. Saltamos.")


## 6. Guardado del dataset limpio

In [ ]:
# Guardamos el dataset con los features añadidos para usarlo en los notebooks 02-05
output_path = ROOT / "data" / "sab_5y_clean.csv"
sab.to_csv(output_path, index=False)
print(f"Guardado: {output_path}")
print(f"Tamaño: {output_path.stat().st_size / 1024:.1f} KB")


## 7. Conclusiones del EDA

Cosas que llevarse al notebook 02 (forecast Prophet):

- La serie tiene una **tendencia clara al alza** desde 2023, acelerada en abril 2024 con el anuncio de la OPA de BBVA.
- La **volatilidad rolling** pega un salto los días clave de noticias OPA — usaremos esos eventos como `holidays` en Prophet.
- **Sin estacionalidad anual evidente** a simple vista; sí semanal (más volumen lunes/viernes) — activamos `weekly_seasonality=True`.
- Los retornos son **leptocúrticos** (colas más gruesas que la normal) — esperar intervalos de confianza amplios.

Siguiente paso → `02_forecast_prophet.ipynb`
